# 一个调用服务端搜索工具的demo

In [ ]:
# Please install OpenAI SDK first: `pip3 install openai`
from openai import OpenAI

client = OpenAI(api_key="xxx", base_url="https://api.deepseek.com")

ws_events = []

with client.responses.stream(
    model="deepseek-v4-flash",
    instructions="You are a helpful assistant.",
    input="Hi, how are you? how's the weather in BEIJING today ?",
    tools=[{
        "type": "web_search",
        "search_context_size": "low",          # low / medium / high
    }]
) as stream:
    for event in stream:
        # 1) 文本增量——直接推给前端
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

        # 2) web_search_call 的生命周期事件
        elif event.type == "response.web_search_call.in_progress":
            ws_events.append({"status": "in_progress", "item_id": event.item_id})
            print(f"\n[搜索启动] item_id={event.item_id}")

        elif event.type == "response.web_search_call.searching":
            ws_events.append({"status": "searching", "item_id": event.item_id})
            print(f"\n[搜索中...] item_id={event.item_id}")

        elif event.type == "response.web_search_call.completed":
            ws_events.append({"status": "completed", "item_id": event.item_id})
            print(f"\n[搜索完成] item_id={event.item_id}")

        # 3) 输出项落地
        elif event.type == "response.output_item.added":
            if event.item.type == "web_search_call":
                print(f"\n[web_search_call 输出项] id={event.item.id}")
                # 注意：流式过程中 action 可能还不完整，
                # 完整 action 要等 final_response

        elif event.type == "response.completed":
            print("\n\n=== 流结束，获取完整 response ===")


I'm doing great, thanks for asking! 😊 Let me check the current weather in Beijing for you right now.
[web_search_call 输出项] id=call_00_dwMavz9vg1HneAfyDSaP9972

[搜索启动] item_id=call_00_dwMavz9vg1HneAfyDSaP9972

[搜索中...] item_id=call_00_dwMavz9vg1HneAfyDSaP9972

[搜索完成] item_id=call_00_dwMavz9vg1HneAfyDSaP9972
Let me get more detailed, up-to-date information from a live weather page.
[web_search_call 输出项] id=call_01_b9RyEH9SSfwZLqE24bYx6438

[搜索启动] item_id=call_01_b9RyEH9SSfwZLqE24bYx6438

[web_search_call 输出项] id=call_02_AKHXE3fItj9euoPEKcii5327

[搜索启动] item_id=call_02_AKHXE3fItj9euoPEKcii5327

[搜索中...] item_id=call_01_b9RyEH9SSfwZLqE24bYx6438

[搜索中...] item_id=call_02_AKHXE3fItj9euoPEKcii5327

[搜索完成] item_id=call_01_b9RyEH9SSfwZLqE24bYx6438

[搜索完成] item_id=call_02_AKHXE3fItj9euoPEKcii5327
Those pages timed out — let me try a couple of other sources.
[web_search_call 输出项] id=call_03_3ZgPLlIunzpqKb5FPwZ72541

[搜索启动] item_id=call_03_3ZgPLlIunzpqKb5FPwZ72541

[web_search_call 输出项] id=call_04